# MLflow's End-to-End RAG Evaluation

This is the official [End-to-End RAG Evaluation](https://mlflow.org/cookbook/rag-evaluation/) with some of my changes (that I may contribute to the official docs later 🤞).

We want to be as much away from Databricks as possible.
This notebook is just a notebook that can be executed on Databricks.

(and perhaps will be until I figure out how to run it on Jupyter Lab).


# Create Project


```shell
uv init --python 3.13 --no-description --no-package mlflow-e2e-rag-eval
```


⚠️ Mind the Python version. **3.13**! 😰 3.14 does not seem to work and fails with the following exception:

<br>

```text
  File "/Users/jacek/demo/mlflow-e2e-rag-eval/.venv/lib/python3.14/site-packages/mlflow/assistant/skill_installer.py", line 11, in <module>
    from importlib.abc import Traversable
```


```shell
uv add "mlflow==3.14.0" openai chromadb
```


```
$ uv tree --depth 1
mlflow-e2e-rag-eval-v3 v0.1.0
├── chromadb v1.5.9
├── mlflow v3.14.0
└── openai v2.43.0
```

# Run MLflow Tracking Server

In a new terminal, run the MLflow Tracking Server.


```shell
uvx --python 3.14 mlflow server --port 5555
```

## From the Sources


Alternatively, if you've got a local clone of the MLflow sources...


```shell
cd ~/oss/mlflow && \
uv build --wheel && \
uvx --python 3.14 --from dist/mlflow-3.14.0-py3-none-any.whl mlflow ui --port 5555
```


The extra `port` argument is to avoid the following issue on mac.

<br>

```console
$ http localhost:5000/api/2.0/mlflow/experiments/get-by-name
HTTP/1.1 403 Forbidden
Content-Length: 0
Server: AirTunes/950.7.1
X-Apple-ProcessingTime: 0
X-Apple-RequestReceivedTimestamp: 36611883
```

⚠️ Note **"Server: AirTunes/950.7.1"** part in the output below.


# Demo App

Create `rag_answer.py` app with the code to verify tracing works (no eval yet).

In [0]:
%skip

import mlflow
import openai
import chromadb

mlflow.set_tracking_uri("http://127.0.0.1:5555")
mlflow.set_experiment("rag-evaluation")

# Create a small knowledge base
docs = [
    "MLflow is an open-source platform for managing the ML lifecycle, including experimentation, reproducibility, and deployment.",
    "MLflow Tracing captures the inputs, outputs, and metadata of each step in a GenAI application, making it easy to debug issues.",
    "MLflow Evaluation uses LLM judges to score the quality of GenAI outputs on metrics like correctness, groundedness, and relevance.",
    "ChromaDB is an open-source vector database for building AI applications with embeddings.",
    "RAG (Retrieval-Augmented Generation) combines a retriever that fetches relevant documents with an LLM that generates answers based on those documents.",
    "MLflow supports over 30 framework integrations including LangChain, LlamaIndex, OpenAI, and Anthropic.",
    "The MLflow AI Gateway provides a unified interface to multiple LLM providers with rate limiting and cost tracking.",
    "MLflow prompt management lets you version, compare, and optimize prompt templates across your applications.",
]

chroma_client = chromadb.Client()
collection = chroma_client.create_collection("knowledge_base")
collection.add(
    documents=docs,
    ids=[f"doc_{i}" for i in range(len(docs))],
)

from mlflow.entities import Document

mlflow.openai.autolog()

client = openai.OpenAI()


@mlflow.trace(span_type="RETRIEVER")
def retrieve(question: str, n_results: int = 3) -> list[Document]:
    results = collection.query(query_texts=[question], n_results=n_results)
    return [
        Document(page_content=doc, metadata={"doc_id": doc_id})
        for doc, doc_id in zip(results["documents"][0], results["ids"][0])
    ]

@mlflow.trace
def rag_answer(question: str) -> str:
    docs = retrieve(question)
    context = "\n".join(doc.page_content for doc in docs)

    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {
                "role": "system",
                "content": f"Answer the question based only on the following context:\n\n{context}",
            },
            {"role": "user", "content": question},
        ],
    )

    return response.choices[0].message.content

answer = rag_answer("Jak zyc z AI pyta Kamil z Warszawy")
print(answer)


```shell
uv run rag_answer.py
```



You should see the following in the MLflow Tracking Server's logs:

<br>

```text
GET /api/2.0/mlflow/experiments/get-by-name?experiment_name=rag-evaluation HTTP/1.1" 404 Not Found
POST /api/2.0/mlflow/experiments/create HTTP/1.1" 200 OK
GET /api/2.0/mlflow/experiments/get?experiment_id=1 HTTP/1.1" 200 OK
GET /version HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces HTTP/1.1" 200 OK
```


Check out the [MLflow UI](http://127.0.0.1:5555).

You should see a trace with spans for rag_answer, retrieve, and the OpenAI chat completion.


Add the eval part to the `rag_answer` app.

In [0]:
%skip

import mlflow
import openai
import chromadb

from mlflow.entities import Document

# Create a small knowledge base
docs = [
    "MLflow is an open-source platform for managing the ML lifecycle, including experimentation, reproducibility, and deployment.",
    "MLflow Tracing captures the inputs, outputs, and metadata of each step in a GenAI application, making it easy to debug issues.",
    "MLflow Evaluation uses LLM judges to score the quality of GenAI outputs on metrics like correctness, groundedness, and relevance.",
    "ChromaDB is an open-source vector database for building AI applications with embeddings.",
    "RAG (Retrieval-Augmented Generation) combines a retriever that fetches relevant documents with an LLM that generates answers based on those documents.",
    "MLflow supports over 30 framework integrations including LangChain, LlamaIndex, OpenAI, and Anthropic.",
    "The MLflow AI Gateway provides a unified interface to multiple LLM providers with rate limiting and cost tracking.",
    "MLflow prompt management lets you version, compare, and optimize prompt templates across your applications.",
]

chroma_client = chromadb.Client()
collection = chroma_client.create_collection("knowledge_base")
collection.add(
    documents=docs,
    ids=[f"doc_{i}" for i in range(len(docs))],
)

# Set up MLflow Client
mlflow.set_tracking_uri("http://127.0.0.1:5555")
mlflow.set_experiment("rag-evaluation")

# Enable OpenAI tracing
mlflow.openai.autolog()

client = openai.OpenAI()


@mlflow.trace(span_type="RETRIEVER")
def retrieve(question: str, n_results: int = 3) -> list[Document]:
    results = collection.query(query_texts=[question], n_results=n_results)
    return [
        Document(page_content=doc, metadata={"doc_id": doc_id})
        for doc, doc_id in zip(results["documents"][0], results["ids"][0])
    ]


@mlflow.trace
def rag_answer(question: str) -> str:
    docs = retrieve(question)
    context = "\n".join(doc.page_content for doc in docs)

    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {
                "role": "system",
                "content": f"Answer the question based only on the following context:\n\n{context}",
            },
            {"role": "user", "content": question},
        ],
    )

    return response.choices[0].message.content

answer = rag_answer("What is MLflow Tracing?")
print(answer)

eval_data = [
    {
        "inputs": {"question": "What is MLflow?"},
        "expectations": {
            "expected_facts": [
                "open-source platform",
                "managing the ML lifecycle",
            ]
        },
    },
    {
        "inputs": {"question": "How does MLflow Tracing work?"},
        "expectations": {
            "expected_facts": [
                "captures inputs and outputs",
                "each step in a GenAI application",
            ]
        },
    },
    {
        "inputs": {"question": "What is RAG?"},
        "expectations": {
            "expected_facts": [
                "retrieval-augmented generation",
                "retriever",
                "LLM",
            ]
        },
    },
    {
        "inputs": {"question": "What vector databases does MLflow integrate with?"},
        "expectations": {
            "expected_facts": ["ChromaDB"],
        },
    },
    {
        "inputs": {"question": "How do you evaluate LLM outputs with MLflow?"},
        "expectations": {
            "expected_facts": [
                "LLM judges",
                "correctness",
                "groundedness",
            ]
        },
    },
]

def predict_fn(question):
    return rag_answer(question)

from mlflow.genai.scorers import (
    RelevanceToQuery,
    RetrievalGroundedness,
    RetrievalSufficiency,
)

results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[
        RelevanceToQuery(),
        RetrievalGroundedness(),
        RetrievalSufficiency(),
    ],
)

# Aggregate metrics
print(results.metrics)

# Example output:
# {'relevance_to_query/mean': 1.0,
#  'retrieval_groundedness/mean': 0.9,
#  'retrieval_sufficiency/mean': 0.8}

# Per-question breakdown
df = results.result_df
print(df[[
    # FIXME there's no column of this name
    # "inputs/question",
    "relevance_to_query/value",
    "retrieval_groundedness/value",
    "retrieval_sufficiency/value",
]])


```shell
uv run full_rag_answer.py
```


You should see the following logs:

<br>

```text
uv run full_rag_answer.py
MLflow Tracing is a feature that captures the inputs, outputs, and metadata of each step in a GenAI application, making it easier to debug issues.
2026/06/23 16:14:46 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/06/23 16:14:46 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [Elapsed: 00:13, Remaining: 00:00] [predict_fn: 21%, scorers: 79%]

✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: agreeable-lamb-728
  Run ID: 9cbb6a1f137a429bac84b2b5c68e03d8
View the evaluation results at http://127.0.0.1:5555/#/experiments/1/evaluation-runs?selectedRunUuid=9cbb6a1f137a429bac84b2b5c68e03d8

{'retrieval_sufficiency/mean': np.float64(0.8), 'retrieval_groundedness/mean': np.float64(1.0), 'relevance_to_query/mean': np.float64(1.0)}
  relevance_to_query/value retrieval_groundedness/value retrieval_sufficiency/value
0                      yes                          yes                         yes
1                      yes                          yes                         yes
2                      yes                          yes                         yes
3                      yes                          yes                          no
4                      yes                          yes                         yes
2026/06/23 16:15:01 INFO mlflow.tracing.export.async_export_queue: Flushing the async trace logging queue before program exit. This may take a while...
```


The MLflow Tracking Server's logs should look similar to the following:

<br>

```text
GET /api/2.0/mlflow/experiments/get-by-name?experiment_name=rag-evaluation HTTP/1.1" 200 OK
POST /api/2.0/mlflow/runs/create HTTP/1.1" 200 OK
POST /api/2.0/mlflow/runs/log-inputs HTTP/1.1" 200 OK
POST /api/2.0/mlflow/runs/set-tag HTTP/1.1" 200 OK
GET /api/2.0/mlflow/runs/get?run_uuid=9cbb6a1f137a429bac84b2b5c68e03d8&run_id=9cbb6a1f137a429bac84b2b5c68e03d8 HTTP/1.1" 200 OK
GET /version HTTP/1.1" 200 OK
GET /version HTTP/1.1" 200 OK
GET /version HTTP/1.1" 200 OK
GET /version HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/tr-7a29a278221f33b07c998ee16acaa21a?trace_id=tr-7a29a278221f33b07c998ee16acaa21a HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/get?trace_id=tr-7a29a278221f33b07c998ee16acaa21a&allow_partial=False HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/tr-81120ab38bb649d63f72358507a2aa09?trace_id=tr-81120ab38bb649d63f72358507a2aa09 HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/get?trace_id=tr-81120ab38bb649d63f72358507a2aa09&allow_partial=False HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/tr-5453bd5d75d2a2412c7bfe7d8310ca9c?trace_id=tr-5453bd5d75d2a2412c7bfe7d8310ca9c HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/get?trace_id=tr-5453bd5d75d2a2412c7bfe7d8310ca9c&allow_partial=False HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/tr-5bd758f79b225962a5c16a2a6feaf319?trace_id=tr-5bd758f79b225962a5c16a2a6feaf319 HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/get?trace_id=tr-5bd758f79b225962a5c16a2a6feaf319&allow_partial=False HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /v1/traces HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/tr-08fa05134d1a26c0ecb31e534652335b?trace_id=tr-08fa05134d1a26c0ecb31e534652335b HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/get?trace_id=tr-08fa05134d1a26c0ecb31e534652335b&allow_partial=False HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-7a29a278221f33b07c998ee16acaa21a/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-7a29a278221f33b07c998ee16acaa21a/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-7a29a278221f33b07c998ee16acaa21a/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-7a29a278221f33b07c998ee16acaa21a/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-5453bd5d75d2a2412c7bfe7d8310ca9c/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-5453bd5d75d2a2412c7bfe7d8310ca9c/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-5453bd5d75d2a2412c7bfe7d8310ca9c/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-5453bd5d75d2a2412c7bfe7d8310ca9c/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-81120ab38bb649d63f72358507a2aa09/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-81120ab38bb649d63f72358507a2aa09/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-81120ab38bb649d63f72358507a2aa09/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-81120ab38bb649d63f72358507a2aa09/assessments HTTP/1.1" 200 OK
GET /api/3.0/mlflow/server-info HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-08fa05134d1a26c0ecb31e534652335b/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-08fa05134d1a26c0ecb31e534652335b/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-08fa05134d1a26c0ecb31e534652335b/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-08fa05134d1a26c0ecb31e534652335b/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-5bd758f79b225962a5c16a2a6feaf319/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-5bd758f79b225962a5c16a2a6feaf319/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-5bd758f79b225962a5c16a2a6feaf319/assessments HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/tr-5bd758f79b225962a5c16a2a6feaf319/assessments HTTP/1.1" 200 OK
POST /api/2.0/mlflow/traces/link-to-run HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/tr-81120ab38bb649d63f72358507a2aa09?trace_id=tr-81120ab38bb649d63f72358507a2aa09 HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/tr-5bd758f79b225962a5c16a2a6feaf319?trace_id=tr-5bd758f79b225962a5c16a2a6feaf319 HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/tr-7a29a278221f33b07c998ee16acaa21a?trace_id=tr-7a29a278221f33b07c998ee16acaa21a HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/tr-5453bd5d75d2a2412c7bfe7d8310ca9c?trace_id=tr-5453bd5d75d2a2412c7bfe7d8310ca9c HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/tr-08fa05134d1a26c0ecb31e534652335b?trace_id=tr-08fa05134d1a26c0ecb31e534652335b HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/get?trace_id=tr-81120ab38bb649d63f72358507a2aa09&allow_partial=False HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/get?trace_id=tr-7a29a278221f33b07c998ee16acaa21a&allow_partial=False HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/get?trace_id=tr-5bd758f79b225962a5c16a2a6feaf319&allow_partial=False HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/get?trace_id=tr-08fa05134d1a26c0ecb31e534652335b&allow_partial=False HTTP/1.1" 200 OK
GET /api/3.0/mlflow/traces/get?trace_id=tr-5453bd5d75d2a2412c7bfe7d8310ca9c&allow_partial=False HTTP/1.1" 200 OK
POST /api/2.0/mlflow/runs/log-batch HTTP/1.1" 200 OK
GET /api/2.0/mlflow/runs/get?run_uuid=9cbb6a1f137a429bac84b2b5c68e03d8&run_id=9cbb6a1f137a429bac84b2b5c68e03d8 HTTP/1.1" 200 OK
POST /api/3.0/mlflow/traces/search HTTP/1.1" 200 OK
GET /api/2.0/mlflow/runs/get?run_uuid=9cbb6a1f137a429bac84b2b5c68e03d8&run_id=9cbb6a1f137a429bac84b2b5c68e03d8 HTTP/1.1" 200 OK
POST /api/2.0/mlflow/runs/update HTTP/1.1" 200 OK
GET /api/2.0/mlflow/runs/get?run_uuid=9cbb6a1f137a429bac84b2b5c68e03d8&run_id=9cbb6a1f137a429bac84b2b5c68e03d8 HTTP/1.1" 200 OK
```


[Navigate to the evaluation run](http://127.0.0.1:5555/#/experiments/1/overview/usage) in the [MLflow UI](http://127.0.0.1:5555).